# Wikidata Non-Article Cleanup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MattArtzAnthro/wikidata-tools/blob/main/notebooks/Wikidata_NonArticle_Cleanup.ipynb)

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook takes your original CrossRef CSV export, identifies non-article entries (Untitled, no authors, journal-issue types, etc.), looks up their DOIs in Wikidata to get their QIDs, and generates deletion commands to remove them.

This approach is more efficient than querying journal-by-journal because you already have the complete source data that was imported. The notebook matches the problematic entries directly by DOI, ensuring you're targeting exactly the items that shouldn't have been imported.

## Key Features

- **Upload CrossRef CSV**: Use your complete CrossRef export
- **Identify Non-Articles**: Filters by Type, author count, and title patterns
- **Batch DOI Lookup**: Query Wikidata for QIDs matching problematic DOIs (100 at a time)
- **Generate Deletions**: Create QuickStatements to blank items or QID lists for deletion requests
- **Track Not Found**: Identifies DOIs that weren't imported (for reference)

## Filter Criteria

Entries are flagged as non-articles if they meet ANY of:
- Type = "journal-issue"
- Author Count = 0
- Title is "Untitled" or empty
- Title equals journal name
- Title < 10 characters

## Workflow

1. **Upload**: Your complete CrossRef CSV export file
2. **Identify**: Apply filters to find problematic entries
3. **Query**: Batch lookup DOIs to get QIDs from the scholarly endpoint
4. **Generate**: Create QuickStatements to remove all properties
5. **Export**: Download CSV report, QuickStatements, and QID list

## Citation

> Artz, M. (2026). Wikidata Tools. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

In [ ]:
# Install required packages
!pip install requests pandas openpyxl ipywidgets -q

import requests
import pandas as pd
import time
import re
import os
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Wikidata endpoints
SCHOLARLY_ENDPOINT = "https://query-scholarly.wikidata.org/sparql"
MAIN_ENDPOINT = "https://query.wikidata.org/sparql"

print("Setup complete. Ready to identify and remove non-articles.")
print("\n⚠️  IMPORTANT: Since May 2025, scholarly articles are on the scholarly endpoint.")

## Upload CrossRef CSV

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Upload file
if IN_COLAB:
    print("Please upload your CrossRef CSV or Excel file...")
    uploaded = files.upload()
else:
    import glob
    csv_files = glob.glob("*.csv") + glob.glob("*.xlsx") + glob.glob("*.xls")
    if csv_files:
        print(f"Found files: {csv_files}")
        print("Set 'filename' variable to the file you want to process.")
    uploaded = None

if uploaded:
    filename = list(uploaded.keys())[0]
elif not uploaded and not IN_COLAB:
    # Set filename manually when running outside Colab
    filename = None  # Set this to your file path
    if filename is None:
        raise ValueError("Set the 'filename' variable to your CSV/Excel file path")

print(f"\nLoaded: {filename}")

# Read the file
if filename.endswith('.csv'):
    df = pd.read_csv(filename, encoding='utf-8-sig')
elif filename.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(filename)
else:
    raise ValueError("File must be CSV or Excel format")

print(f"Loaded {len(df):,} records")
print(f"\nColumns: {', '.join(df.columns[:10])}...")

## Identify Non-Articles in CrossRef Data

In [ ]:
def identify_non_articles(df):
    """
    Identify non-article entries using the same logic as CrossRef_Article_Filter.
    """
    total = len(df)
    df['filter_reason'] = None
    flags = []
    
    # Filter 1: journal-issue types
    if 'Type' in df.columns:
        mask = df['Type'] == 'journal-issue'
        df.loc[mask, 'filter_reason'] = 'Type: journal-issue'
        flags.append(('journal-issue', mask.sum()))
    
    # Filter 2: no authors
    if 'Author Count' in df.columns:
        mask = df['Author Count'] == 0
        df.loc[mask & df['filter_reason'].isna(), 'filter_reason'] = 'No authors'
        flags.append(('no authors', mask.sum()))
    
    # Filter 3: Untitled
    if 'Title' in df.columns:
        mask = df['Title'] == 'Untitled'
        df.loc[mask & df['filter_reason'].isna(), 'filter_reason'] = 'Untitled'
        flags.append(('Untitled', mask.sum()))
    
    # Filter 4: Title = Journal
    if 'Title' in df.columns and 'Journal' in df.columns:
        mask = df['Title'] == df['Journal']
        df.loc[mask & df['filter_reason'].isna(), 'filter_reason'] = 'Title = Journal'
        flags.append(('Title = Journal', mask.sum()))
    
    # Filter 5: Short titles
    if 'Title' in df.columns:
        mask = df['Title'].fillna('').str.len() < 10
        df.loc[mask & df['filter_reason'].isna(), 'filter_reason'] = 'Short title'
        flags.append(('Short title', mask.sum()))
    
    # Return only flagged items
    non_articles = df[df['filter_reason'].notna()].copy()
    
    return non_articles, flags

print("Analyzing CrossRef data for non-article patterns...\n")
non_articles, flags = identify_non_articles(df)

print("="*70)
print("ANALYSIS RESULTS")
print("="*70)
print(f"\nTotal CrossRef records: {len(df):,}")
print(f"Flagged as non-articles: {len(non_articles):,}")
print(f"\nBreakdown by filter:")
for filter_name, count in flags:
    if count > 0:
        print(f"  {filter_name}: {count:,}")

if len(non_articles) > 0:
    print(f"\n{'='*70}")
    print("SAMPLE NON-ARTICLES")
    print("="*70)
    display_cols = ['Title', 'DOI', 'Type', 'Author Count', 'filter_reason']
    display_cols = [col for col in display_cols if col in non_articles.columns]
    display(non_articles[display_cols].head(10))
else:
    print("\n✓ No non-articles detected!")

## Clean and Normalize DOIs

In [ ]:
def clean_doi(doi_input):
    """Normalize DOI to standard format (uppercase, no URL prefix)."""
    if not doi_input or not isinstance(doi_input, str):
        return None
    
    doi_input = str(doi_input).strip()
    
    # Extract DOI from URLs
    for pattern in [r'https?://(?:dx\.)?doi\.org/(.+)', r'doi:(.+)']:
        match = re.search(pattern, doi_input, re.IGNORECASE)
        if match:
            doi_input = match.group(1)
            break
    
    # Validate DOI format
    if re.match(r'^10\.\d+/.+', doi_input):
        return doi_input.strip().upper()  # Wikidata stores DOIs uppercase
    
    return None

if len(non_articles) > 0 and 'DOI' in non_articles.columns:
    non_articles['doi_clean'] = non_articles['DOI'].apply(clean_doi)
    
    with_doi = non_articles['doi_clean'].notna().sum()
    without_doi = non_articles['doi_clean'].isna().sum()
    
    print(f"DOI Status:")
    print(f"  Non-articles with DOI: {with_doi:,}")
    print(f"  Non-articles without DOI: {without_doi:,}")
    
    if without_doi > 0:
        print(f"\n⚠️  {without_doi} non-articles have no DOI and cannot be looked up in Wikidata.")
    
    # Keep only items with DOIs for Wikidata lookup
    non_articles_with_doi = non_articles[non_articles['doi_clean'].notna()].copy()
    print(f"\n✓ {len(non_articles_with_doi):,} non-articles ready for Wikidata lookup")
else:
    non_articles_with_doi = pd.DataFrame()
    print("No non-articles to process.")

## Batch Query Wikidata for QIDs

In [ ]:
def batch_query_dois(doi_list, batch_size=100):
    """
    Query Wikidata for QIDs matching a list of DOIs.
    Processes in batches to avoid query timeouts.
    """
    all_results = []
    
    for i in range(0, len(doi_list), batch_size):
        batch = doi_list[i:i+batch_size]
        
        # Build VALUES clause for SPARQL
        values_clause = ' '.join([f'("{doi}")' for doi in batch])
        
        query = f"""
        SELECT ?article ?doi WHERE {{
          VALUES (?doi) {{ {values_clause} }}
          ?article wdt:P356 ?doi .
        }}
        """
        
        try:
            response = requests.get(
                SCHOLARLY_ENDPOINT,
                params={'query': query, 'format': 'json'},
                headers={'User-Agent': 'WikidataNonArticleCleanup/1.0 (matt@mattartz.me)'},
                timeout=60
            )
            response.raise_for_status()
            data = response.json()
            
            for item in data['results']['bindings']:
                qid = item['article']['value'].split('/')[-1]
                doi = item['doi']['value']
                all_results.append({'doi': doi, 'qid': qid})
            
            print(f"  Batch {i//batch_size + 1}: Found {len([r for r in all_results if r['doi'] in batch])} / {len(batch)} DOIs")
            
        except Exception as e:
            print(f"  Error in batch {i//batch_size + 1}: {e}")
        
        time.sleep(1)  # Rate limiting
    
    return pd.DataFrame(all_results)

if len(non_articles_with_doi) > 0:
    print(f"Querying Wikidata for {len(non_articles_with_doi):,} DOIs...")
    print(f"Processing in batches of 100...\n")
    
    doi_list = non_articles_with_doi['doi_clean'].tolist()
    results_df = batch_query_dois(doi_list)
    
    print(f"\n{'='*70}")
    print("WIKIDATA LOOKUP RESULTS")
    print("="*70)
    print(f"\nDOIs searched: {len(doi_list):,}")
    print(f"QIDs found in Wikidata: {len(results_df):,}")
    print(f"DOIs not found: {len(doi_list) - len(results_df):,}")
    
    # Merge QIDs back to non_articles
    non_articles_with_qid = non_articles_with_doi.merge(
        results_df,
        left_on='doi_clean',
        right_on='doi',
        how='left'
    )
    
    found = non_articles_with_qid['qid'].notna().sum()
    not_found = non_articles_with_qid['qid'].isna().sum()
    
    print(f"\nMatched to non-articles:")
    print(f"  Found in Wikidata: {found:,}")
    print(f"  Not in Wikidata: {not_found:,}")
    
    if not_found > 0:
        print(f"\n⚠️  {not_found} DOIs were not found. They may not have been imported,")
        print("   or they may be on the main endpoint instead of scholarly endpoint.")
    
    # Show sample
    print(f"\n{'='*70}")
    print("SAMPLE WIKIDATA MATCHES")
    print("="*70)
    display_cols = ['qid', 'Title', 'doi_clean', 'filter_reason']
    display_cols = [col for col in display_cols if col in non_articles_with_qid.columns]
    display(non_articles_with_qid[non_articles_with_qid['qid'].notna()][display_cols].head(10))
else:
    non_articles_with_qid = pd.DataFrame()
    print("No items to query.")

## Generate Deletion QuickStatements

In [ ]:
def get_all_claims(qid):
    """
    Retrieve all claims (properties) for a given Wikidata item.
    """
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        if qid in data['entities']:
            claims = data['entities'][qid].get('claims', {})
            return claims
        return {}
    except Exception as e:
        return {}

def generate_deletion_quickstatements(items_df):
    """
    Generate QuickStatements V1 commands to remove all statements from items.
    """
    if len(items_df) == 0:
        return []
    
    statements = []
    statements.append("# QuickStatements to blank non-article items")
    statements.append(f"# Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    statements.append(f"# Source: {filename}")
    statements.append(f"# Items to blank: {len(items_df)}")
    statements.append("#")
    statements.append("# WARNING: This will remove ALL properties from these items!")
    statements.append("# Review carefully before running.")
    statements.append("#")
    statements.append("")
    
    processed = 0
    for idx, row in items_df.iterrows():
        qid = row['qid']
        title = row.get('Title', 'No title')
        doi = row.get('doi_clean', 'No DOI')
        reason = row.get('filter_reason', 'Unknown')
        
        statements.append(f"# {qid} - {title[:50]} - DOI: {doi}")
        statements.append(f"# Reason: {reason}")
        
        # Get all claims
        claims = get_all_claims(qid)
        
        if claims:
            for property_id in claims.keys():
                statements.append(f"{qid}|-{property_id}")
        else:
            statements.append(f"# Could not fetch claims for {qid}")
        
        statements.append("")
        processed += 1
        
        if processed % 10 == 0:
            print(f"  Processed {processed}/{len(items_df)} items...")
        
        time.sleep(0.3)  # Rate limiting
    
    return statements

if len(non_articles_with_qid) > 0:
    items_to_delete = non_articles_with_qid[non_articles_with_qid['qid'].notna()].copy()
    
    if len(items_to_delete) > 0:
        print(f"\nGenerating QuickStatements to blank {len(items_to_delete)} items...")
        print("This will take a while as we fetch each item's properties...\n")
        
        quickstatements = generate_deletion_quickstatements(items_to_delete)
        
        print(f"\n✓ Generated {len(quickstatements)} QuickStatements lines")
        print("\nPreview:")
        print("\n".join(quickstatements[:25]))
        if len(quickstatements) > 25:
            print(f"\n... ({len(quickstatements) - 25} more lines)")
    else:
        quickstatements = []
        print("\nNo items found in Wikidata to delete.")
else:
    quickstatements = []
    print("No items to process.")

## Export Results

In [ ]:
# Create output directory
if IN_COLAB:
    output_dir = "/content/outputs"
else:
    output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
base_name = os.path.splitext(filename)[0]

files_created = []

# Export all non-articles (with and without QIDs)
if len(non_articles_with_qid) > 0:
    csv_file = f"{output_dir}/{base_name}_NonArticles_{timestamp}.csv"
    export_cols = ['qid', 'Title', 'DOI', 'doi_clean', 'Type', 'Author Count', 
                   'filter_reason', 'Journal', 'Year']
    export_cols = [col for col in export_cols if col in non_articles_with_qid.columns]
    non_articles_with_qid[export_cols].to_csv(csv_file, index=False)
    files_created.append(csv_file)
    print(f"Exported non-articles CSV: {csv_file}")

# Export QuickStatements
if quickstatements:
    qs_file = f"{output_dir}/{base_name}_Deletion_QS_{timestamp}.txt"
    with open(qs_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(quickstatements))
    files_created.append(qs_file)
    print(f"Exported QuickStatements: {qs_file}")

# Export QID list
if len(non_articles_with_qid) > 0:
    items_with_qid = non_articles_with_qid[non_articles_with_qid['qid'].notna()]
    if len(items_with_qid) > 0:
        qid_file = f"{output_dir}/{base_name}_QIDs_to_Delete_{timestamp}.txt"
        with open(qid_file, 'w') as f:
            f.write(f"# Non-article items to delete\n")
            f.write(f"# Source: {filename}\n")
            f.write(f"# Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"# Total items: {len(items_with_qid)}\n\n")
            for qid in items_with_qid['qid']:
                f.write(f"{qid}\n")
        files_created.append(qid_file)
        print(f"Exported QID list: {qid_file}")

# Export summary
summary_file = f"{output_dir}/{base_name}_Summary_{timestamp}.txt"
with open(summary_file, 'w') as f:
    f.write("Wikidata Non-Article Cleanup Summary\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Source file: {filename}\n")
    f.write(f"Processing date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"Total CrossRef records: {len(df):,}\n")
    f.write(f"Non-articles identified: {len(non_articles):,}\n")
    if len(non_articles_with_doi) > 0:
        f.write(f"Non-articles with DOI: {len(non_articles_with_doi):,}\n")
    if len(non_articles_with_qid) > 0:
        found = non_articles_with_qid['qid'].notna().sum()
        f.write(f"Found in Wikidata: {found:,}\n")
        f.write(f"Not found in Wikidata: {len(non_articles_with_qid) - found:,}\n")
    f.write("\nFilter breakdown:\n")
    for filter_name, count in flags:
        if count > 0:
            f.write(f"  {filter_name}: {count:,}\n")
files_created.append(summary_file)
print(f"Exported summary: {summary_file}")

# Export items not found in Wikidata (for reference)
if len(non_articles_with_qid) > 0:
    not_found = non_articles_with_qid[non_articles_with_qid['qid'].isna()]
    if len(not_found) > 0:
        not_found_file = f"{output_dir}/{base_name}_NotFound_{timestamp}.csv"
        export_cols = ['Title', 'DOI', 'doi_clean', 'Type', 'filter_reason']
        export_cols = [col for col in export_cols if col in not_found.columns]
        not_found[export_cols].to_csv(not_found_file, index=False)
        files_created.append(not_found_file)
        print(f"Exported not-found list: {not_found_file}")

print(f"\n{'='*70}")
print("SUMMARY")
print("="*70)
if len(non_articles_with_qid) > 0:
    items_to_delete = non_articles_with_qid[non_articles_with_qid['qid'].notna()]
    print(f"\n{len(items_to_delete):,} non-articles ready to delete from Wikidata")
    print(f"{len(files_created)} files exported")
else:
    print("\nNo items to delete.")

print(f"\n{'='*70}")
print("NEXT STEPS")
print("="*70)
print("\n1. Review the CSV to confirm these are non-articles")
print("2. Run the QuickStatements file to blank all properties:")
print("   - Go to https://quickstatements.toolforge.org/")
print("   - Upload the .txt file or paste contents")
print("   - Review and run")
print("\n3. Or submit QID list to Wikidata:Requests for deletion")

## Download Files

In [ ]:
if files_created:
    print("Downloading files...\n")
    for file_path in files_created:
        print(f"  {os.path.basename(file_path)}")
        if IN_COLAB:
            files.download(file_path)
    if IN_COLAB:
        print("\nAll files downloaded!")
    else:
        print(f"\nFiles saved to: {output_dir}/")
else:
    print("No files to download.")